In [7]:
import pickle
import pandas as pd
import numpy as np
from pathlib import Path

BASE = "/data/rbg/users/ashyamal/RNA-FM-benchmarking"

BAD_VALUES = {"unknown", "not reported", "'--", "not applicable", "n/a", "na", "none", "", "nan/null", "--"}

def is_bad(x):
    if pd.isnull(x):
        return True
    return str(x).strip().lower() in BAD_VALUES

def availability(df):
    """Return DataFrame with % available and n unique non-bad values per column."""
    def col_stats(col):
        good_mask = ~col.map(is_bad)
        return pd.Series({
            "pct_available": good_mask.mean(),
            "n_unique":      col[good_mask].nunique(),
        })
    return df.apply(col_stats).T.sort_values("pct_available", ascending=False)

def collect_split_ids(split_dir, n_splits=5):
    """Return all unique sample IDs across all splits (train + test)."""
    ids = set()
    for i in range(n_splits):
        p = Path(split_dir) / f"split_{i}.pkl"
        if p.exists():
            s = pickle.load(open(p, "rb"))
            ids.update(s["train"] + s["test"])
    return ids


In [8]:
# TCGA: sample IDs == cases.submitter_id
split_ids_tcga = collect_split_ids(f"{BASE}/splits/TCGA_cleaned")

clin_tcga = pd.read_csv(f"{BASE}/data/TCGA/clinical/clinical.tsv", sep="\t", low_memory=False)
clin_tcga = clin_tcga.drop_duplicates(subset="cases.submitter_id").set_index("cases.submitter_id")
clin_tcga = clin_tcga.loc[clin_tcga.index.isin(split_ids_tcga)]
print(f"TCGA — {len(clin_tcga)} patients in splits, {clin_tcga.shape[1]} columns")

df_tcga = availability(clin_tcga)
df_tcga["n_available"] = (df_tcga["pct_available"] * len(clin_tcga)).round(0).astype(int)
df_tcga["pct_available"] = (df_tcga["pct_available"] * 100).round(1)
df_tcga["n_unique"] = df_tcga["n_unique"].astype(int)

display(df_tcga[df_tcga["pct_available"] > 0]
        .style.background_gradient(subset="pct_available", cmap="YlGn"))

TCGA — 10386 patients in splits, 186 columns


,pct_available,n_unique,n_available
cases.case_id,100.000000,10386,10386
project.project_id,100.000000,33,10386
demographic.gender,100.000000,2,10386
demographic.submitter_id,100.000000,10386,10386
demographic.age_is_obfuscated,100.000000,2,10386
demographic.demographic_id,100.000000,10386,10386
diagnoses.diagnosis_is_primary_disease,100.000000,2,10386
diagnoses.diagnosis_id,100.000000,10386,10386
diagnoses.submitter_id,100.000000,10386,10386
follow_ups.updated_datetime,100.000000,33,10386


In [9]:
# CPTAC: sample IDs == cases.case_id
split_ids_cptac = collect_split_ids(f"{BASE}/splits/CPTAC")

clin_cptac = pd.read_csv("/data/rbg/shared/datasets/CPTAC/clinical/clinical.tsv", sep="\t", low_memory=False)
clin_cptac = clin_cptac.drop_duplicates(subset="cases.case_id").set_index("cases.case_id")
clin_cptac = clin_cptac.loc[clin_cptac.index.isin(split_ids_cptac)]
print(f"CPTAC — {len(clin_cptac)} patients in splits, {clin_cptac.shape[1]} columns")

df_cptac = availability(clin_cptac)
df_cptac["n_available"] = (df_cptac["pct_available"] * len(clin_cptac)).round(0).astype(int)
df_cptac["pct_available"] = (df_cptac["pct_available"] * 100).round(1)
df_cptac["n_unique"] = df_cptac["n_unique"].astype(int)

display(df_cptac[df_cptac["pct_available"] > 0]
        .style.background_gradient(subset="pct_available", cmap="YlGn"))

CPTAC — 1222 patients in splits, 209 columns


,pct_available,n_unique,n_available
project.project_id,100.000000,1,1222
cases.consent_type,100.000000,1,1222
cases.days_to_consent,100.000000,118,1222
cases.disease_type,100.000000,7,1222
cases.index_date,100.000000,1,1222
cases.submitter_id,100.000000,1222,1222
cases.primary_site,100.000000,16,1222
demographic.days_to_birth,100.000000,1170,1222
demographic.age_is_obfuscated,100.000000,2,1222
diagnoses.days_to_last_follow_up,100.000000,827,1222


In [10]:
# TARGET_AML: strip last suffix from sample ID to get cases.submitter_id
split_ids_target = collect_split_ids(f"{BASE}/splits/TARGET_AML")
patient_ids_target = {"-".join(s.rsplit("-", 1)[:1]) for s in split_ids_target}

clin_target = pd.read_csv(f"{BASE}/data/TARGET_AML/clinical/clinical.tsv", sep="\t", low_memory=False)
clin_target = clin_target.drop_duplicates(subset="cases.submitter_id").set_index("cases.submitter_id")
clin_target = clin_target.loc[clin_target.index.isin(patient_ids_target)]
print(f"TARGET_AML — {len(clin_target)} patients in splits, {clin_target.shape[1]} columns")

df_target = availability(clin_target)
df_target["n_available"] = (df_target["pct_available"] * len(clin_target)).round(0).astype(int)
df_target["pct_available"] = (df_target["pct_available"] * 100).round(1)
df_target["n_unique"] = df_target["n_unique"].astype(int)

display(df_target[df_target["pct_available"] > 0]
        .style.background_gradient(subset="pct_available", cmap="YlGn"))

TARGET_AML — 1902 patients in splits, 209 columns


,pct_available,n_unique,n_available
project.project_id,100.000000,1,1902
cases.case_id,100.000000,1902,1902
cases.primary_site,100.000000,1,1902
cases.index_date,100.000000,1,1902
cases.disease_type,100.000000,1,1902
demographic.age_is_obfuscated,100.000000,1,1902
demographic.days_to_birth,100.000000,1644,1902
demographic.age_at_index,100.000000,30,1902
diagnoses.diagnosis_id,100.000000,1902,1902
diagnoses.classification_of_tumor,100.000000,1,1902


In [11]:
# Hartwig: sample IDs == sampleId in metadata
split_ids_hartwig = collect_split_ids(f"{BASE}/splits/Hartwig")

meta_hartwig = pd.read_csv(
    "/data/rbg/shared/datasets/Hartwig/isofox_tpm_update_4/metadata.tsv", sep="\t"
)
meta_hartwig = meta_hartwig.drop_duplicates(subset="sampleId").set_index("sampleId")
meta_hartwig = meta_hartwig.loc[meta_hartwig.index.isin(split_ids_hartwig)]
print(f"Hartwig — {len(meta_hartwig)} samples in splits, {meta_hartwig.shape[1]} columns")

df_hartwig = availability(meta_hartwig)
df_hartwig["n_available"] = (df_hartwig["pct_available"] * len(meta_hartwig)).round(0).astype(int)
df_hartwig["pct_available"] = (df_hartwig["pct_available"] * 100).round(1)
df_hartwig["n_unique"] = df_hartwig["n_unique"].astype(int)

display(df_hartwig[df_hartwig["pct_available"] > 0]
        .style.background_gradient(subset="pct_available", cmap="YlGn"))

Hartwig — 2231 samples in splits, 23 columns


,pct_available,n_unique,n_available
specimenId,100.000000,2231,2231
patientId,100.000000,2152,2231
treatmentStartDate,100.000000,887,2231
hasRadiotherapyPreTreatment,100.000000,2,2231
hasSystemicPreTreatment,100.000000,4,2231
doids,100.000000,128,2231
biopsyDate,100.000000,846,2231
treatmentGiven,100.000000,2,2231
treatment,100.000000,482,2231
consolidatedTreatmentType,100.000000,9,2230
